# Oppitunti 17 - Paikallisten AI-agenttien luominen Foundry Localilla ja Qwenillä

Tässä muistikirjassa rakennat **paikallisen insinööriavustajan**, joka toimii täysin työasemallasi. Pilvipohjaista päättelyä ei käytetä lainkaan. Avustaja voi:

1. **Kutsua työkaluja** Qwen-funktiokutsun kautta Foundry Localilla.
2. **Luetteloida ja lukea tiedostoja** hiekkalaatikkomuotoisessa projektihakemistossa.
3. **Analysoida koodia** yksinkertaisilla mittareilla.
4. **Etsiä dokumentaatiota** paikallisella RAG:llä (Chroma).
5. **Käyttää paikallista MCP-palvelinta** (ohitetaan tyylikkäästi, jos sitä ei ole konfiguroitu).

Agenttikoodi näyttää lähes identtiseltä pilvioppituntien kanssa — ainoa ero on, että asiakasliittymän päätepiste siirtyy pilvestä `localhostiin`.


## Asennus

Ennen tämän muistikirjan suorittamista:

1. **Asenna Microsoft Foundry Local** (katso [dokumentaatio](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) omalle käyttöjärjestelmällesi).
2. **Lataa ja käynnistä Qwen-malli:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Asenna alla olevat Python-kirjastot.

Kaikki toimii paikallisesti. Kone, jossa on noin 8 Gt RAM-muistia, on realistinen vähimmäisvaatimus.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Yhdistä Foundry Localiin

`FoundryLocalManager` lataa mallin tarvittaessa, käynnistää paikallisen palvelun ja antaa meille **OpenAI-yhteensopivan päätepisteen**. Kohdistamme sitten tavallisen OpenAI SDK:n siihen. API-avain on paikallinen paikkamerkki — mitään pilvivaltuutusta ei ole mukana.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Paikalliset työkalut (hiekkalaatikossa eristetyt tiedostotoiminnot)

Luomme levylle pienen malliprojektin ja määrittelemme työkalut, jotka rajoittuvat tuon projektin juureen. Hiekkalaatikkotarkistuksella on merkitystä myös paikallisesti: työkalu, joka lukee mielivaltaisia polkuja, toimii käyttäjäsi oikeuksilla ja voi käsitellä mitä tahansa, mitä sinäkin voit.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Paikallinen RAG Chroman kanssa

Upotamme pienen joukon dokumentaatiopätkiä paikalliseen Chroma-kokoelmaan. Chroma suoritetaan samassa prosessissa ja tallentaa vektorit kiintolevylle — ei palvelinta, ei pilveä. `search_docs`-työkalu hakee haun kannalta merkityksellisimmät pätkät.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Työkalukutsusilmukka

Nyt rekisteröimme työkalut malliin käyttäen OpenAI-työkalukaaviota ja suoritetaan vakio työkalu-kutsusilmukka — malli pyytää työkalua, me suoritamme sen paikallisesti, syötämme tuloksen takaisin, ja toistamme, kunnes malli tuottaa lopullisen vastauksen. Qwenin luotettava funktiokutsu on se, mikä saa tämän toimimaan laitteella.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Paikallinen MCP (valinnainen)

MCP on tiedonsiirtokerros, ei pilvipalvelu — MCP-palvelin voi toimia paikallisena prosessina `stdio`-yhteyden yli. Alla oleva solu näyttää, kuinka yhdistät paikalliseen MCP-palvelimeen, jos sinulla on sellainen konfiguroituna (esimerkiksi tiedostojärjestelmäpalvelin). Se ohittaa toiminnon tyylikkäästi, jos `LOCAL_MCP_COMMAND` ei ole asetettu, joten muistikirja toimii yhä kokonaisuudessaan ilman sitä.

Turvallisuusmuistutus: paikallinen MCP-palvelin toimii käyttäjäsi oikeuksilla. Rajoita se projektihakemistoon ja varmista sen tuottamat tulokset ennen niihin perustuvia toimia.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Yhteenveto

Rakensit insinööriavustajan, joka toimii kokonaan omalla koneellasi:

- **Foundry Local** tarjosi **Qwen**-mallin OpenAI-yhteensopivan päätepisteen takana — joten agenttikoodi vastaa pilviharjoituksia.
- **Hiekkalaatikkotyökalut** antoivat agentille tiedostojen käyttöoikeuden ja koodin analyysin ilman, että projektihakemiston ulkopuolelle siirryttiin.
- **Chroma** tarjosi **paikallisen RAG:n** dokumentaation yli.
- **Local MCP** osoitti, miten MCP-ekosysteemiä voi käyttää uudelleen offline-tilassa.

Pilvilaskennan ennustetta (inference) ei käytetty missään vaiheessa.

### Haaste

Laajenna paikallista agenttia seuraavasti:

1. **Tukea useita MCP-palvelimia** — yhdistä tiedostojärjestelmäpalvelin ja git-palvelin ja anna agentin valita niiden välillä.
2. **Käyttää paikallista muistia** — tallenna lyhyt keskusteluhistoria levylle niin, että avustaja muistaa aiemmat vuorot muistikirjan uudelleenkäynnistyksissä.
3. **Tukea paikallista multi-agenttien orkestrointia** — lisää toinen paikallinen agentti (esim. arvioija) ja salli heidän tehdä yhteistyötä tehtävän parissa.

Seuraavassa oppitunnissa opit, miten suojata käyttöön otetut tekoälyagentit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
